# Heart Disease Prediction V2: Leakage Audit and Cross-Hospital Validation

V1 (2022) compared Random Forest, XGBoost and a neural network on a merged heart disease file and reported accuracy only. This notebook re-does the work with a stricter protocol:

1. **Leakage audit**: how much do duplicate rows in the widely shared 1,025-row file inflate scores?
2. **Internal validation**: repeated stratified 5-fold cross-validation on the four UCI hospitals, duplicates removed.
3. **Leave-one-hospital-out**: train on three hospitals, test on the fourth (Cleveland, Hungary, Switzerland, VA Long Beach).
4. **Models**: logistic regression, Random Forest, XGBoost, LightGBM, CatBoost and the TabPFN tabular foundation model.
5. **Beyond AUROC**: calibration (ECE, calibration slope/intercept), bootstrap 95% CIs, SHAP, and subgroups by sex and age.

**Settings before running:** Accelerator = None (CPU is enough), Internet = On. For the leakage audit, add the Kaggle dataset `johnsmith88/heart-disease-dataset` with *Add Input*. Then *Save Version* → *Save & Run All*.

Code: https://github.com/MHKhan50/heart-disease · Research only, not for diagnosis.

In [ ]:
# Install the project package and TabPFN (XGBoost, LightGBM, CatBoost and SHAP are preinstalled on Kaggle)
!pip install -q --no-deps git+https://github.com/MHKhan50/heart-disease.git
!pip install -q tabpfn || echo "TabPFN could not be installed; it will be skipped"

In [ ]:
from pathlib import Path
import warnings
import pandas as pd

from heartv2.data import FEATURES_ALL, FEATURES_CORE, drop_duplicates, load_kaggle_merged, load_uci, site_summary
from heartv2.evaluate import internal_cv, leakage_audit, leave_one_site_out, loso_table, subgroup_auroc, summarise_cv
from heartv2.models import get_models
from heartv2 import plots

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
SEED, N_REPEATS, N_BOOT = 42, 5, 1000
OUT = Path("/kaggle/working/results") if Path("/kaggle/working").exists() else Path("results")
(OUT / "figures").mkdir(parents=True, exist_ok=True)

## 1. Data

The four processed UCI files are downloaded from the UCI repository. A cholesterol or resting blood pressure of 0 is recoded as missing (the Switzerland and VA files use 0 for "not measured"). The target is angiographic disease (`num > 0`).

In [ ]:
df_raw = load_uci()
df, n_dup = drop_duplicates(df_raw)
print(f"{len(df_raw)} rows loaded, {n_dup} exact duplicates removed, {len(df)} used")
site_summary(df)

## 2. Leakage audit

The 1,025-row file circulated on Kaggle contains repeated records. With a random 80/20 split, many test rows also appear in training, so the model is partly tested on data it has already seen. The audit repeats the split 20 times with and without duplicates (Random Forest, same settings).

In [ ]:
merged = load_kaggle_merged()
if merged is None:
    print("Add the dataset 'johnsmith88/heart-disease-dataset' as input to run the audit.")
else:
    audit = leakage_audit(merged, n_repeats=20, seed=SEED)
    audit.to_csv(OUT / "leakage_audit.csv")
    display(audit)

## 3. Models

Hyperparameters are fixed in advance (sensible defaults, no tuning on test data). Preprocessing (imputation with missing-value indicators, scaling, one-hot encoding) is fitted inside every training fold.

In [ ]:
models = get_models(SEED)

## 4. Internal validation (pooled, repeated 5-fold CV)

In [ ]:
folds, oof = internal_cv(df, models, FEATURES_ALL, n_repeats=N_REPEATS, seed=SEED)
folds.to_csv(OUT / "internal_cv_folds.csv", index=False)
summary = summarise_cv(folds)
summary.to_csv(OUT / "internal_cv_summary.csv")
summary

## 5. Leave-one-hospital-out

Two feature sets: **all** 13 attributes, and **core** (without `slope`, `ca`, `thal`, which are mostly missing outside Cleveland). AUROC with patient-level bootstrap 95% CI.

In [ ]:
loso, loso_preds = {}, {}
for name, feats in {"all": FEATURES_ALL, "core": FEATURES_CORE}.items():
    print(f"Feature set: {name}")
    res, preds = leave_one_site_out(df, models, feats, n_boot=N_BOOT, seed=SEED)
    loso[name], loso_preds[name] = res.assign(features=name), preds
    loso_table(res).to_csv(OUT / f"loso_auroc_{name}.csv")
pd.concat(loso.values()).to_csv(OUT / "loso_all_metrics.csv", index=False)
display(loso_table(loso["all"]))
display(loso_table(loso["core"]))

In [ ]:
cols = ["held_out", "model", "n", "prevalence", "auroc", "brier", "ece", "cal_intercept", "cal_slope", "sensitivity", "specificity"]
loso["all"][cols].round(3)

In [ ]:
best = loso["all"].groupby("model")["auroc"].mean().idxmax()
print("Best mean external AUROC:", best)
plots.internal_vs_external(folds, loso["all"], OUT / "figures/internal_vs_external.png");
plots.calibration_curves(df, loso_preds["all"], best, path=OUT / "figures/calibration_loso.png");

## 6. Explainability (SHAP) and subgroups

In [ ]:
tree = next(m for m in ("XGBoost", "LightGBM", "RandomForest") if m in models)
plots.shap_summary(df, models[tree], FEATURES_ALL, path=OUT / "figures/shap_summary.png", seed=SEED);

In [ ]:
sub = subgroup_auroc(df, oof[best], n_boot=N_BOOT, seed=SEED)
sub.to_csv(OUT / "subgroups.csv", index=False)
sub.round(3)

## 7. Limitations

- The data were collected in the 1980s at four hospitals; the target is angiographic disease, not a modern clinical endpoint.
- Switzerland has very few patients without disease, so its AUROC interval is wide and its calibration is dominated by prevalence.
- Hyperparameters are not tuned; the comparison is between reasonable defaults under the same protocol.
- Research code only, not for diagnosis.

In [ ]:
sorted(str(p.relative_to(OUT)) for p in OUT.rglob("*") if p.is_file())